# 21 — Rapid DEV Triage Fusion & Calibration

## Purpose

Time is limited. This notebook does **no new training**.

It takes the strongest saved ICH experiments and rapidly searches for a better final triage pipeline on the common DEV split:

- 2D Blood + Expanded True Negatives
- 2.5D Blood + Expanded True Negatives
- continuity-aware ICH post-processing
- 2D / 2.5D volume fusion
- MLS threshold-bin calibration
- fracture threshold calibration

The search is staged and DEV-only to reduce overfitting and runtime.

The locked TEST split is **not used for tuning** in this notebook.

## 1. Imports

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 300)

## 2. Configuration

In [ ]:
ICH_CLASSES = ["EDH", "SDH", "IPH", "SAH", "IVH"]
ICH_COLUMNS = [f"V_{name}" for name in ICH_CLASSES]

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]
OUTPUT_ROOT = Path("/kaggle/working/rapid_dev_triage_fusion")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ALPHAS = np.arange(0.0, 1.01, 0.1)
FRACTURE_THRESHOLDS = np.arange(0.20, 0.81, 0.02)

MLS_T1_GRID = np.arange(0.25, 2.01, 0.25)
MLS_T3_GRID = np.arange(1.50, 5.01, 0.25)
MLS_T5_GRID = np.arange(3.00, 8.01, 0.25)

ICH_PROFILES = {
    "raw": {"EDH": 1, "SDH": 1, "IPH": 1, "SAH": 1, "IVH": 1},
    "all2": {"EDH": 2, "SDH": 2, "IPH": 2, "SAH": 2, "IVH": 2},
    "all3": {"EDH": 3, "SDH": 3, "IPH": 3, "SAH": 3, "IVH": 3},
    "soft_sah1": {"EDH": 2, "SDH": 2, "IPH": 2, "SAH": 1, "IVH": 2},
    "soft_sah2": {"EDH": 2, "SDH": 2, "IPH": 2, "SAH": 2, "IVH": 2},
    "edh2_sdh2_iph2_sah1_ivh3": {"EDH": 2, "SDH": 2, "IPH": 2, "SAH": 1, "IVH": 3},
}

print("Output:", OUTPUT_ROOT)

## 3. Auto-discover saved Notebook 17 / 18 outputs

Attach the **saved outputs** of Notebook 17 and Notebook 18 to this notebook.

The code identifies them by reading `00_DIRECT_ANSWERS.csv`, so the Kaggle dataset names do not need to match a hard-coded path.

In [ ]:
def unique_paths(paths):
    result = []
    seen = set()
    for path in paths:
        key = str(path.resolve())
        if key not in seen:
            seen.add(key)
            result.append(path)
    return result

def find_all(filename):
    matches = []
    for root in SEARCH_ROOTS:
        if root.exists():
            matches.extend(root.rglob(filename))
    return unique_paths(matches)

def identify_experiment_roots():
    found = {}
    for path in find_all("00_DIRECT_ANSWERS.csv"):
        try:
            table = pd.read_csv(path)
        except Exception:
            continue

        text = " ".join(table.astype(str).fillna("").values.ravel()).lower()

        if "2.5d" in text and "expanded" in text:
            found["2p5d"] = path.parent
        elif "2d blood" in text and "expanded" in text and "2.5d" not in text:
            found["2d"] = path.parent

    return found

experiment_roots = identify_experiment_roots()

if "2d" not in experiment_roots or "2p5d" not in experiment_roots:
    print("Discovered experiment roots:", experiment_roots)
    raise FileNotFoundError("Attach saved outputs of both Notebook 17 and Notebook 18.")

ROOT_2D = experiment_roots["2d"]
ROOT_2P5D = experiment_roots["2p5d"]

print("2D root:", ROOT_2D)
print("2.5D root:", ROOT_2P5D)

## 4. Locate the required DEV prediction files

In [ ]:
def find_under(root, filename):
    matches = list(root.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected exactly one {filename} below {root}, found {len(matches)}.")
    return matches[0]

SERIES_2D_PATH = find_under(ROOT_2D, "04_dev_series_predictions.csv")
SLICES_2D_PATH = find_under(ROOT_2D, "dev_slice_predictions.csv")
SERIES_2P5D_PATH = find_under(ROOT_2P5D, "04_dev_series_predictions.csv")
SLICES_2P5D_PATH = find_under(ROOT_2P5D, "dev_slice_predictions.csv")

final_eval_matches = find_all("common_dev_predictions.csv")
preferred = [path for path in final_eval_matches if "final_evaluation" in str(path)]
if len(preferred) == 1:
    BASE_DEV_PATH = preferred[0]
elif len(final_eval_matches) == 1:
    BASE_DEV_PATH = final_eval_matches[0]
else:
    print("common_dev_predictions.csv matches:")
    for path in final_eval_matches:
        print(" -", path)
    raise FileNotFoundError("Attach the saved final-evaluation output.")

print("2D series:", SERIES_2D_PATH)
print("2.5D series:", SERIES_2P5D_PATH)
print("Base DEV:", BASE_DEV_PATH)

## 5. Load tables

In [ ]:
def normalize_series_id(value):
    try:
        return str(int(float(value)))
    except Exception:
        return str(value).strip()

def load_series(path):
    df = pd.read_csv(path).copy()
    df["series_id"] = df["series_id"].map(normalize_series_id)
    return df

def load_slices(path):
    df = pd.read_csv(path).copy()
    df["series_id"] = df["series_id"].map(normalize_series_id)
    return df

series_2d = load_series(SERIES_2D_PATH)
slices_2d = load_slices(SLICES_2D_PATH)
series_2p5d = load_series(SERIES_2P5D_PATH)
slices_2p5d = load_slices(SLICES_2P5D_PATH)
base_dev = pd.read_csv(BASE_DEV_PATH).copy()
base_dev["series_id"] = base_dev["series_id"].map(normalize_series_id)

common_ids = set(base_dev["series_id"]) & set(series_2d["series_id"]) & set(series_2p5d["series_id"])
if len(common_ids) != 54:
    raise RuntimeError(f"Expected 54 common DEV series, found {len(common_ids)}.")

base_dev = base_dev[base_dev["series_id"].isin(common_ids)].copy().sort_values("series_id").reset_index(drop=True)
series_2d = series_2d[series_2d["series_id"].isin(common_ids)].copy()
series_2p5d = series_2p5d[series_2p5d["series_id"].isin(common_ids)].copy()

print("DEV series:", len(common_ids))

## 6. Official triage rule

In [ ]:
def triage_from_values(values):
    V_EDH = max(0.0, float(values["V_EDH"]))
    V_SDH = max(0.0, float(values["V_SDH"]))
    V_IPH = max(0.0, float(values["V_IPH"]))
    V_SAH = max(0.0, float(values["V_SAH"]))
    V_IVH = max(0.0, float(values["V_IVH"]))
    fracture_prob = float(values["fracture_prob"])
    MLS_mm = max(0.0, float(values["MLS_mm"]))
    total_vol = V_EDH + V_SDH + V_IPH + V_SAH + V_IVH
    has_ich = total_vol >= 0.1
    fracture_present = fracture_prob >= 0.5

    if MLS_mm >= 5.0 and (has_ich or fracture_present): return 2
    if V_EDH >= 30.0: return 2
    if V_SDH >= 70.0: return 2
    if V_IPH >= 70.0: return 2
    if total_vol >= 60.0: return 2
    if has_ich and MLS_mm >= 3.0 and total_vol >= 40.0: return 2
    if fracture_present and total_vol >= 15.0: return 2
    if MLS_mm >= 5.0 and not (has_ich or fracture_present): return 1
    if has_ich: return 1
    if 3.0 <= MLS_mm < 5.0: return 1
    if fracture_present and total_vol < 15.0: return 1
    if total_vol >= 0.1 and MLS_mm >= 1.0: return 1
    return 0

def macro_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, average="macro", labels=[0, 1, 2], zero_division=0))

def evaluate_pipeline(ich_df, mls_values, fracture_values):
    merged = base_dev[["series_id", "true_triage"]].merge(ich_df, on="series_id", how="inner")
    merged["MLS_mm"] = np.asarray(mls_values, dtype=float)
    merged["fracture_prob"] = np.asarray(fracture_values, dtype=float)

    predictions = []
    for row in merged.itertuples(index=False):
        values = {column: float(getattr(row, column)) for column in ICH_COLUMNS}
        values["MLS_mm"] = float(row.MLS_mm)
        values["fracture_prob"] = float(row.fracture_prob)
        predictions.append(triage_from_values(values))

    return {
        "macro_F1": macro_f1(merged["true_triage"], predictions),
        "accuracy": float(accuracy_score(merged["true_triage"], predictions)),
        "predictions": np.asarray(predictions, dtype=int),
    }

## 7. Continuity-aware ICH series volumes

In [ ]:
def keep_runs(group, subtype, min_run):
    group = group.sort_values("slice_order")
    flags = group[f"pred_positive_{subtype}"].astype(bool).to_numpy()
    volumes = group[f"pred_volume_{subtype}"].to_numpy(dtype=float)
    keep = np.zeros(len(group), dtype=bool)
    start = None

    for index in range(len(flags) + 1):
        active = index < len(flags) and flags[index]

        if active and start is None:
            start = index

        if not active and start is not None:
            if index - start >= min_run:
                keep[start:index] = True
            start = None

    return float(volumes[keep].sum())

def aggregate_profile(slice_df, profile):
    rows = []

    for series_id, group in slice_df.groupby("series_id", sort=False):
        row = {"series_id": series_id}
        for subtype in ICH_CLASSES:
            row[f"V_{subtype}"] = keep_runs(group, subtype, profile[subtype])
        rows.append(row)

    return pd.DataFrame(rows)

ich_candidates = {}

for model_name, slice_df in [("2d", slices_2d), ("2p5d", slices_2p5d)]:
    for profile_name, profile in ICH_PROFILES.items():
        key = f"{model_name}__{profile_name}"
        ich_candidates[key] = aggregate_profile(slice_df, profile)

print("ICH candidate tables:", len(ich_candidates))

## 8. Stage A — choose ICH profile and 2D/2.5D fusion

In [ ]:
current_mls = base_dev["pred_MLS_mm"].to_numpy(dtype=float)
current_fracture = base_dev["pred_fracture_prob"].to_numpy(dtype=float)

ich_search_rows = []

profile_names = list(ICH_PROFILES)

for profile_2d in profile_names:
    left = ich_candidates[f"2d__{profile_2d}"]

    for profile_2p5d in profile_names:
        right = ich_candidates[f"2p5d__{profile_2p5d}"]
        merged = left.merge(right, on="series_id", suffixes=("_2d", "_2p5d"))

        for alpha in ALPHAS:
            fused = pd.DataFrame({"series_id": merged["series_id"]})

            for column in ICH_COLUMNS:
                fused[column] = alpha * merged[f"{column}_2d"] + (1.0 - alpha) * merged[f"{column}_2p5d"]

            result = evaluate_pipeline(fused, current_mls, current_fracture)
            ich_search_rows.append({
                "profile_2d": profile_2d,
                "profile_2p5d": profile_2p5d,
                "alpha_2d": float(alpha),
                "alpha_2p5d": float(1.0 - alpha),
                "macro_F1": result["macro_F1"],
                "accuracy": result["accuracy"],
            })

ich_search_df = pd.DataFrame(ich_search_rows).sort_values(["macro_F1", "accuracy"], ascending=False).reset_index(drop=True)
ich_search_df.to_csv(OUTPUT_ROOT / "01_ich_fusion_search.csv", index=False)

display(ich_search_df.head(20))

## 9. Freeze the best ICH candidate

In [ ]:
best_ich = ich_search_df.iloc[0]

left = ich_candidates[f"2d__{best_ich['profile_2d']}"]
right = ich_candidates[f"2p5d__{best_ich['profile_2p5d']}"]
merged = left.merge(right, on="series_id", suffixes=("_2d", "_2p5d"))

best_ich_df = pd.DataFrame({"series_id": merged["series_id"]})
for column in ICH_COLUMNS:
    best_ich_df[column] = float(best_ich["alpha_2d"]) * merged[f"{column}_2d"] + float(best_ich["alpha_2p5d"]) * merged[f"{column}_2p5d"]

print("Best ICH configuration:")
display(best_ich.to_frame().T)

## 10. Stage B — calibrate MLS into triage-relevant bins

In [ ]:
def calibrate_mls(values, t1, t3, t5):
    values = np.asarray(values, dtype=float)
    output = np.zeros(len(values), dtype=float)
    output[(values >= t1) & (values < t3)] = 1.5
    output[(values >= t3) & (values < t5)] = 3.5
    output[values >= t5] = 5.5
    return output

mls_search_rows = []

for t1 in MLS_T1_GRID:
    for t3 in MLS_T3_GRID:
        if t3 <= t1:
            continue

        for t5 in MLS_T5_GRID:
            if t5 <= t3:
                continue

            calibrated = calibrate_mls(current_mls, t1, t3, t5)
            result = evaluate_pipeline(best_ich_df, calibrated, current_fracture)

            mls_search_rows.append({
                "t1": float(t1),
                "t3": float(t3),
                "t5": float(t5),
                "macro_F1": result["macro_F1"],
                "accuracy": result["accuracy"],
            })

mls_search_df = pd.DataFrame(mls_search_rows).sort_values(["macro_F1", "accuracy"], ascending=False).reset_index(drop=True)
mls_search_df.to_csv(OUTPUT_ROOT / "02_mls_bin_calibration_search.csv", index=False)

display(mls_search_df.head(20))

## 11. Freeze MLS calibration

In [ ]:
best_mls = mls_search_df.iloc[0]
best_mls_values = calibrate_mls(current_mls, best_mls["t1"], best_mls["t3"], best_mls["t5"])

print("Best MLS calibration:")
display(best_mls.to_frame().T)

## 12. Stage C — fracture threshold calibration

In [ ]:
fracture_search_rows = []

for threshold in FRACTURE_THRESHOLDS:
    calibrated_fracture = (current_fracture >= threshold).astype(float)
    result = evaluate_pipeline(best_ich_df, best_mls_values, calibrated_fracture)

    fracture_search_rows.append({
        "threshold": float(threshold),
        "macro_F1": result["macro_F1"],
        "accuracy": result["accuracy"],
    })

fracture_search_df = pd.DataFrame(fracture_search_rows).sort_values(["macro_F1", "accuracy"], ascending=False).reset_index(drop=True)
fracture_search_df.to_csv(OUTPUT_ROOT / "03_fracture_threshold_search.csv", index=False)

display(fracture_search_df.head(20))

## 13. Final DEV pipeline

In [ ]:
best_fracture = fracture_search_df.iloc[0]
best_fracture_values = (current_fracture >= float(best_fracture["threshold"])).astype(float)

baseline_ich = series_2p5d[["series_id"] + ICH_COLUMNS].copy()
baseline = evaluate_pipeline(baseline_ich, current_mls, current_fracture)
final_result = evaluate_pipeline(best_ich_df, best_mls_values, best_fracture_values)

summary_df = pd.DataFrame([
    {
        "configuration": "2.5D raw + current MLS + current fracture",
        "macro_F1": baseline["macro_F1"],
        "accuracy": baseline["accuracy"],
    },
    {
        "configuration": "DEV-optimized fusion + MLS bins + fracture threshold",
        "macro_F1": final_result["macro_F1"],
        "accuracy": final_result["accuracy"],
    },
])

summary_df.to_csv(OUTPUT_ROOT / "00_DIRECT_ANSWERS.csv", index=False)
display(summary_df)

print("Selected 2D profile:", best_ich["profile_2d"])
print("Selected 2.5D profile:", best_ich["profile_2p5d"])
print("2D weight:", float(best_ich["alpha_2d"]))
print("2.5D weight:", float(best_ich["alpha_2p5d"]))
print("MLS thresholds:", float(best_mls["t1"]), float(best_mls["t3"]), float(best_mls["t5"]))
print("Fracture threshold:", float(best_fracture["threshold"]))

## 14. Save deployable calibration config

In [ ]:
config = {
    "ich": {
        "profile_2d": str(best_ich["profile_2d"]),
        "profile_2p5d": str(best_ich["profile_2p5d"]),
        "alpha_2d": float(best_ich["alpha_2d"]),
        "alpha_2p5d": float(best_ich["alpha_2p5d"]),
        "profile_rules": ICH_PROFILES,
    },
    "mls": {
        "predicted_threshold_for_true_1mm_bin": float(best_mls["t1"]),
        "predicted_threshold_for_true_3mm_bin": float(best_mls["t3"]),
        "predicted_threshold_for_true_5mm_bin": float(best_mls["t5"]),
        "representative_outputs_mm": [0.0, 1.5, 3.5, 5.5],
    },
    "fracture": {
        "threshold": float(best_fracture["threshold"]),
    },
    "dev_macro_F1": float(final_result["macro_F1"]),
    "dev_accuracy": float(final_result["accuracy"]),
    "locked_test_used_for_tuning": False,
}

with open(OUTPUT_ROOT / "rapid_calibration_config.json", "w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)

print("Saved:", OUTPUT_ROOT / "rapid_calibration_config.json")

# What to do next

If this notebook produces a substantial DEV gain, use its frozen configuration in the inference pipeline.

Because time is limited, do **not** start another large architecture sweep before checking this result. The fastest remaining gains are likely to come from:

1. better ICH fusion / continuity,
2. MLS bin calibration,
3. only then a fracture threshold adjustment.

After this configuration is frozen, the next step is to inject it into the final `model.py` / inference package and verify end-to-end runtime and model size.